# Homework 3

In [203]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer


In [204]:
df = pd.read_csv("./course_lead_scoring_2026.csv")
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


# Question 1

In [205]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
Name: count, dtype: int64

# Question 2

In [206]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [207]:
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns
categorical_columns = df.select_dtypes(include=['str']).columns
numerical_columns, categorical_columns

(Index(['annual_income', 'number_of_courses_viewed', 'interaction_count',
        'lead_score', 'converted'],
       dtype='str'),
 Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='str'))

In [208]:
df[numerical_columns] = df[numerical_columns].fillna(0, inplace=True)
df[categorical_columns] =df[categorical_columns].fillna('NA', inplace=True)
df.isnull().sum()

/var/folders/z5/ff_brn5x5yd9y6687m3g92c80000gn/T/ipykernel_5645/1300878399.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[numerical_columns] = df[numerical_columns].fillna(0, inplace=True)
/var/folders/z5/ff_brn5x5yd9y6687m3g92c80000gn/T/ipykernel_5645/1300878399.py:2: ChainedAssignmentError: A value is being set on a copy of

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [209]:
correlation_matrix = df[numerical_columns].corr()
correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


# Question 3

In [210]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [211]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

df_train_final = df_train.drop(columns='converted')
df_val_final = df_val.drop(columns='converted')
df_test_final = df_test.drop(columns='converted')

In [212]:
def mutual_information_score(series, df = df_train):
    return mutual_info_score(series, df.converted)


(df_train[categorical_columns].apply(mutual_information_score)
    .round(2)
    .sort_values(ascending=False)
)


lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

# Question 4

In [213]:
dv = DictVectorizer(sparse=False)
train_dicts = df_train_final.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val_final.to_dict(orient='records')
X_val = dv.transform(val_dicts)

test_dicts = df_test_final.to_dict(orient='records')
X_test = dv.transform(test_dicts)

In [214]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

original_accuracy = accuracy_score(y_val, y_pred)
print(f'Validation Accuracy: {round(original_accuracy, 2)}')

Validation Accuracy: 0.65


# Question 5

In [215]:
def prepare_train_model_and_get_accuracy(df_train_final, df_val_final, y_train, y_val, columns_to_use):
    dv = DictVectorizer(sparse=False)
    train_dicts = df_train_final[columns_to_use].to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    val_dicts = df_val_final[columns_to_use].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)

    return accuracy

In [219]:
for column in df_train_final.columns:
    column_to_use = df_train_final.drop(columns=[column]).columns
    accuracy = prepare_train_model_and_get_accuracy(df_train_final, df_val_final, y_train, y_val, column_to_use)
    print(f'Validation Accuracy without column {column}: {accuracy}; difference from original: {original_accuracy - accuracy}')


Validation Accuracy without column lead_source: 0.642; difference from original: 0.0030000000000000027
Validation Accuracy without column industry: 0.645; difference from original: 0.0
Validation Accuracy without column employment_status: 0.645; difference from original: 0.0
Validation Accuracy without column location: 0.645; difference from original: 0.0
Validation Accuracy without column annual_income: 0.724; difference from original: -0.07899999999999996
Validation Accuracy without column number_of_courses_viewed: 0.643; difference from original: 0.0020000000000000018
Validation Accuracy without column interaction_count: 0.601; difference from original: 0.04400000000000004
Validation Accuracy without column lead_score: 0.644; difference from original: 0.0010000000000000009


# Question 6

In [217]:
def get_accuracy_with_regularized_logistic(df_train_final, df_val_final, y_train, y_val, C=1.0):
    dv = DictVectorizer(sparse=False)
    train_dicts = df_train_final.to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    val_dicts = df_val_final.to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)

    return accuracy

In [221]:
Cs = [0.000001, 0.00001, 0.0001, 0.001]
for C in Cs:
    accuracy = get_accuracy_with_regularized_logistic(df_train_final, df_val_final, y_train, y_val, C=C)
    print(f'Validation Accuracy with C={C}: {round(accuracy, 3)}')

Validation Accuracy with C=1e-06: 0.598
Validation Accuracy with C=1e-05: 0.598
Validation Accuracy with C=0.0001: 0.613
Validation Accuracy with C=0.001: 0.645
